In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
project_pth=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)
from utils.transformations import Dynamic



In [0]:
%sql select * from spotify_catalog.gold.dimtrack
where track_id in (5,46)

In [0]:
df1=spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
    .option('cloudFiles.schemaLocation','abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimUser/schema')\
    .option('cloudFiles.schemaEvolutionMode','addNewColumns')\
    .load('abfss://bronze@r1pipelinestorage.dfs.core.windows.net/DimUser/')



In [0]:
df_user=df1.withColumn('user_name',upper(col('user_name')))

In [0]:

df_user_obj=Dynamic()
df_user=df_user_obj.dropColumns(df1,*['_rescued_data'])

df_user=df1.dropDuplicates(['user_id'])


In [0]:
query = (
    df_user.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimUser/checkpoint"
    )
    .outputMode("append")
    .trigger(once=True)
    .start(
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimUser/data"
    )
)

In [0]:
df2=spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
    .option('cloudFiles.schemaLocation','abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimArt/schema')\
    .option('cloudFiles.schemaEvolutionMode','addNewColumns')\
    .load('abfss://bronze@r1pipelinestorage.dfs.core.windows.net/DimArtist/')



In [0]:
display(df2)

In [0]:
df_user_obj=Dynamic()
df_user=df_user_obj.dropColumns(df2,*['_rescued_data'])

df_user=df2.dropDuplicates(['user_id'])

In [0]:
query = (
    df_user.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimArt/checkpoint"
    )
    .outputMode("append")
    .trigger(once=True)
    .start(
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimArt/data"
    )
)

In [0]:
df2 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimArt/schema"
    )
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(
        "abfss://bronze@r1pipelinestorage.dfs.core.windows.net/DimArtist/"
    )
)

df_artist_obj = Dynamic()

df_artist = df_artist_obj.dropColumns(
    df2,
    *["_rescued_data"]
)

df_artist = df_artist.dropDuplicates(["artist_id"])

query = (
    df_artist.writeStream
    .format("delta").option("mergeSchema", "true")
    .option(
        "checkpointLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimArt/checkpoint"
    )
    .outputMode("append")
    .trigger(once=True)
    .toTable("spotify_catalog.silver.DimArt")
)

query.awaitTermination()




In [0]:
from pyspark.sql.functions import when, col

df_track = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimTrack/schema"
    )
    .load(
        "abfss://bronze@r1pipelinestorage.dfs.core.windows.net/DimTrack/"
    )
)

# Transformation: create duration_flag
df_track = df_track.withColumn(
    "duration_flag",
    when(col("duration_sec") < 150, "low")
    .when(col("duration_sec") < 300, "medium")
    .otherwise("high")
)

df_track=df_track.withColumn('track_name',regexp_replace(col('track_name'),'-',''))

df_track=Dynamic().dropColumns(df_track,*['_rescued_data'])


# Write to Silver ADLS + Unity Catalog table
query = (
    df_track.writeStream
    .format("delta").option("mergeSchema", "true")
    .option(
        "checkpointLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimTrack/checkpoint"
    )
    .outputMode("append")
    .trigger(once=True)
    .toTable("spotify_catalog.silver.DimTrack")
)

query.awaitTermination()

In [0]:
# Read from Bronze using Auto Loader
df_date = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimDate/schema"
    )
    .load(
        "abfss://bronze@r1pipelinestorage.dfs.core.windows.net/DimDate/"
    )
)

# Transformation: remove rescued column
df_date = Dynamic().dropColumns(
    df_date,
    *["_rescued_data"]
)

# Write to ADLS + Unity Catalog
query = (
    df_date.writeStream
    .format("delta")
    .option("mergeSchema", "true")
    .option(
        "checkpointLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimDate/checkpoint"
    )
    .outputMode("append")
    .trigger(once=True)
    .toTable("spotify_catalog.silver.DimDate")
)

query.awaitTermination()

In [0]:
df_fact=spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
    .option('cloudFiles.schemaLocation','abfss://silver@r1pipelinestorage.dfs.core.windows.net/FactStream/schema')\
    .option('cloudFiles.schemaEvolutionMode','addNewColumns')\
    .load('abfss://bronze@r1pipelinestorage.dfs.core.windows.net/FactStream/')


In [0]:
df_fact=Dynamic().dropColumns(df_fact,*['_rescued_data'])

In [0]:
query = (
    df_fact.writeStream
    .format("delta")
    .option("mergeSchema", "true").option(
        "checkpointLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/FactStream/checkpoint"
    )
    .outputMode("append")
    .trigger(once=True)
    .toTable("spotify_catalog.silver.FactStream")
)

query.awaitTermination()

In [0]:
from pyspark.sql.functions import upper, col

# Read from Bronze using Auto Loader
df1 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimUser/schema"
    )
    .load(
        "abfss://bronze@r1pipelinestorage.dfs.core.windows.net/DimUser/"
    )
)

# Transformation 1: Convert user_name to uppercase
df_user = df1.withColumn(
    "user_name",
    upper(col("user_name"))
)

# Transformation 2: Remove _rescued_data
df_user_obj = Dynamic()

df_user = df_user_obj.dropColumns(
    df_user,
    *["_rescued_data"]
)

# Transformation 3: Remove duplicate users
df_user = df_user.dropDuplicates(
    ["user_id"]
)

# Write Silver data to ADLS + Unity Catalog
query = (
    df_user.writeStream
    .format("delta")
    .option("mergeSchema", "true")
    .option(
        "checkpointLocation",
        "abfss://silver@r1pipelinestorage.dfs.core.windows.net/DimUser/checkpoint"
    )
    .outputMode("append")
    .trigger(once=True)
    .toTable(
        "spotify_catalog.silver.DimUser"
    )
)

query.awaitTermination()